# 06 · HarmBench generations: base vs. intervention

Greedy generations (256 new tokens) for HarmBench text behaviors with and without the top-4 refusal-latent intervention. Copyright behaviors are excluded, which leaves 300 behaviors (200 standard + 100 contextual). The output CSVs contain empty `manual_label` / `manual_note` columns for manual refusal labelling.

> Generations of the intervened model can contain harmful content. They are written only to your Drive and are intentionally not part of this repository.

In [ ]:
# !pip install -q -U transformers accelerate pandas tqdm huggingface_hub safetensors

from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive')
HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()

from pathlib import Path
from contextlib import contextmanager, nullcontext
import json
import random
import subprocess

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'meta-llama/Meta-Llama-3-8B-Instruct'
SAE_OUT_DIR = Path('/content/drive/MyDrive/Diploma/sae_runs/k56')
SAE_CKPT_PATH = SAE_OUT_DIR / 'checkpoints' / 'sae_epoch_025.pt'
EVAL_OUT_DIR = Path('/content/drive/MyDrive/Diploma/sae_eval_k56_refusal_features')
FEATURE_OUT_DIR = EVAL_OUT_DIR / 'refusal_features_strict_retain005'
SELECTED_FEATURES_JSON_PATH = FEATURE_OUT_DIR / 'selected_refusal_features_strict.json'
HB_ROOT = Path('/content/HarmBench')
HB_BEHAVIORS_PATH = HB_ROOT / 'data/behavior_datasets/harmbench_behaviors_text_all.csv'
HB_RESULTS_DIR = EVAL_OUT_DIR / 'harmbench_generation_top4'
HB_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

D_IN = 4096
D_SAE = 16384
K = 56
HOOK_LAYER = 12
TARGET_FEATURE_TOP_N = 4
CLAMP_VALUE = 0.0
MAX_NEW_TOKENS = 256
HARM_PROMPT_MAX_LENGTH = 1024
BATCH_SIZE = 1
DO_SAMPLE = False
GEN_USE_CACHE = True
VARIANTS = ['base', 'sae_intervention']
SAE_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

if not HB_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/centerforaisafety/HarmBench.git', str(HB_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(HB_ROOT), 'pull'], check=False)

In [ ]:
def read_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def get_hidden_from_output(output):
    return output[0] if isinstance(output, tuple) else output


def replace_hidden_in_output(output, new_hidden):
    if isinstance(output, tuple):
        output = list(output)
        output[0] = new_hidden
        return tuple(output)
    return new_hidden


class TopKSAE(nn.Module):
    def __init__(self, d_in: int, d_sae: int, k: int, dtype: torch.dtype):
        super().__init__()
        self.W_enc = nn.Parameter(torch.empty(d_in, d_sae, dtype=dtype))
        self.W_dec = nn.Parameter(torch.empty(d_sae, d_in, dtype=dtype))
        self.b_enc = nn.Parameter(torch.zeros(d_sae, dtype=dtype))
        self.b_dec = nn.Parameter(torch.zeros(d_in, dtype=dtype))
        self.k = k
        self.d_sae = d_sae

    def encode_topk(self, x: torch.Tensor):
        pre_acts = (x - self.b_dec) @ self.W_enc + self.b_enc
        topk_vals, topk_idx = pre_acts.topk(self.k, dim=-1)
        return F.relu(topk_vals), topk_idx

    def encode_dense(self, x: torch.Tensor):
        topk_vals, topk_idx = self.encode_topk(x)
        feat = torch.zeros(x.shape[0], self.d_sae, device=x.device, dtype=x.dtype)
        feat.scatter_(-1, topk_idx, topk_vals)
        return feat

In [ ]:
sae = TopKSAE(D_IN, D_SAE, K, SAE_DTYPE).to(DEVICE)
sae.load_state_dict(torch.load(SAE_CKPT_PATH, map_location=DEVICE, weights_only=True)['model_state_dict'])
sae.eval()
TARGET_FEATURES = [int(x) for x in read_json(SELECTED_FEATURES_JSON_PATH)['selected_features'][:TARGET_FEATURE_TOP_N]]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=MODEL_DTYPE, device_map='auto', low_cpu_mem_usage=True, attn_implementation='sdpa')
model.eval()
model.config.use_cache = GEN_USE_CACHE
INPUT_DEVICE = next(model.model.embed_tokens.parameters()).device

In [ ]:
@torch.inference_mode()
def apply_delta_intervention_to_flat(x_flat: torch.Tensor) -> torch.Tensor:
    feat = sae.encode_dense(x_flat)
    delta = torch.zeros_like(x_flat)
    for j in TARGET_FEATURES:
        active = feat[:, j] > 0
        if active.any():
            delta[active] += (CLAMP_VALUE - feat[active, j]).unsqueeze(-1) * sae.W_dec[j].unsqueeze(0)
    return x_flat + delta


def patch_hidden(hidden: torch.Tensor) -> torch.Tensor:
    b, s, d = hidden.shape
    x_flat = hidden.reshape(-1, d).to(DEVICE, dtype=SAE_DTYPE)
    patched = apply_delta_intervention_to_flat(x_flat)
    return patched.reshape(b, s, d).to(hidden.device, dtype=hidden.dtype)


@contextmanager
def sae_patch_context():
    def hook(module, inputs, output):
        hidden = get_hidden_from_output(output)
        return replace_hidden_in_output(output, patch_hidden(hidden))
    handle = model.model.layers[HOOK_LAYER].register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()


def generation_context(variant: str):
    if variant == 'base':
        return nullcontext()
    if variant == 'sae_intervention':
        return sae_patch_context()
    raise ValueError(variant)


def render_chat(user_prompt: str, system_prompt: str = 'You are a helpful assistant.') -> str:
    messages = [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': user_prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def build_prompt(row: pd.Series) -> str:
    behavior = str(row['Behavior']).strip()
    context = row.get('ContextString', '')
    return context.strip() + '\n\n' + behavior if isinstance(context, str) and context.strip() else behavior


@torch.inference_mode()
def generate_batch(prompts: list[str], variant: str) -> list[str]:
    texts = [render_chat(prompt) for prompt in prompts]
    batch = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=HARM_PROMPT_MAX_LENGTH, add_special_tokens=False)
    batch = {k: v.to(INPUT_DEVICE) for k, v in batch.items()}
    with generation_context(variant):
        output_ids = model.generate(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            use_cache=GEN_USE_CACHE,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return [tokenizer.decode(row[batch['input_ids'].shape[1]:], skip_special_tokens=True).strip() for row in output_ids]


def generate_harmbench_variant(variant: str, behaviors_df: pd.DataFrame) -> pd.DataFrame:
    json_path = HB_RESULTS_DIR / f'completions_{variant}_top4.json'
    csv_path = HB_RESULTS_DIR / f'completions_{variant}_top4_rows.csv'
    completions = read_json(json_path) if json_path.exists() else {}
    items = []
    for _, row in behaviors_df.iterrows():
        behavior_id = str(row['BehaviorID'])
        if behavior_id not in completions:
            items.append((behavior_id, row, build_prompt(row)))

    for start in tqdm(range(0, len(items), BATCH_SIZE), desc=f'HarmBench | {variant}'):
        batch_items = items[start:start + BATCH_SIZE]
        generations = generate_batch([x[2] for x in batch_items], variant)
        for (behavior_id, row, prompt), generation in zip(batch_items, generations):
            completions[behavior_id] = [{
                'test_case': prompt,
                'generation': generation,
                'Behavior': row.get('Behavior', ''),
                'FunctionalCategory': row.get('FunctionalCategory', ''),
                'SemanticCategory': row.get('SemanticCategory', ''),
            }]
        with json_path.open('w', encoding='utf-8') as f:
            json.dump(completions, f, ensure_ascii=False, indent=2)

    rows = []
    for behavior_id, values in completions.items():
        for item in values:
            rows.append({
                'variant': variant,
                'BehaviorID': behavior_id,
                'Behavior': item.get('Behavior', ''),
                'FunctionalCategory': item.get('FunctionalCategory', ''),
                'SemanticCategory': item.get('SemanticCategory', ''),
                'test_case': item.get('test_case', ''),
                'generation': item.get('generation', ''),
                'manual_label': '',
                'manual_note': '',
            })
    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    return df

In [ ]:
HB_FUNCTIONAL_CATEGORIES = ['standard', 'contextual']
harmbench_df = pd.read_csv(HB_BEHAVIORS_PATH)
harmbench_df = harmbench_df[harmbench_df['FunctionalCategory'].isin(HB_FUNCTIONAL_CATEGORIES)].reset_index(drop=True)
print('HarmBench behaviors:', len(harmbench_df))
harmbench_outputs = {variant: generate_harmbench_variant(variant, harmbench_df) for variant in VARIANTS}
display(pd.concat(harmbench_outputs.values(), ignore_index=True).head())